# Overfitting and cross-validation

Data Scientist track, lesson DS5. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** see overfitting as a gap between the score on the fitting rows and on unseen rows, run k-fold cross-validation and read its spread, choose stratified or time-based folds to fit the question, and keep every step that learns inside the folds.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, TimeSeriesSplit, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 2026       # the seed of the whole Data Scientist track

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")

## The split and two models

The columns and the split of lesson DS2 (the same test part, scored on in this lesson only in Exercise 3), the day-7 model of lesson DS2, and a decision tree whose flexibility is one number, its depth (lesson DS7 opens it up). Run this cell as it is.

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)


def logit():
    """The day-7 model of lesson DS2: one-hot text columns, scaled number columns, logistic regression."""
    return make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", StandardScaler(), WEEK1)]), LogisticRegression(max_iter=1000))


def tree(depth, leaf=1):
    """A decision tree of a given depth: the deeper it is allowed to grow, the more it can memorize."""
    return make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", "passthrough", WEEK1)]),
                         DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, random_state=SEED))


folds = StratifiedKFold(5, shuffle=True, random_state=SEED)       # the five folds used from unit 2 on
print(len(X_train), len(X_test))

## Unit 1: Overfitting is a gap

**Predict first.** As the depth grows, the AUC on the fitting rows keeps rising. Will the AUC on the validation rows keep rising too, rise and then fall, or stay flat? At about what depth does it stop improving?

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train)   # a validation part, cut from the training part
curve = []
for depth in [1, 2, 3, 4, 6, 8, 12, None]:                                   # None: let the tree grow until it cannot split any more
    m = tree(depth).fit(X_fit, y_fit)
    curve.append(("no limit" if depth is None else depth, round(float(roc_auc_score(y_fit, m.predict_proba(X_fit)[:, 1])), 3), round(float(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])), 3)))
print(pd.DataFrame(curve, columns=["max_depth", "AUC on the fitting rows", "AUC on the validation rows"]).to_string(index=False))

## Unit 2: K-fold: a score and its spread

**Predict first.** Will the five fold scores of the day-7 model all lie within 0.01 of each other, within 0.03, or further apart? And will it beat the depth-3 tree on every fold?

In [ ]:
scores = cross_val_score(logit(), X_train, y_train, cv=folds, scoring="roc_auc")      # five fits, each scored on the fold it did not learn from
print(np.round(scores, 3), round(float(scores.mean()), 3), round(float(scores.std()), 4))
tree_scores = cross_val_score(tree(3), X_train, y_train, cv=folds, scoring="roc_auc")  # the same five folds for a second model
print(round(float(tree_scores.mean()), 3), np.round(scores - tree_scores, 3))

## Unit 3: Folds that fit the question

**Predict first.** The model will be used on later signups. Will its mean AUC on four time-based folds be lower than the 0.692 of the random folds, about the same, or higher?

In [ ]:
Xs, ys = X_train.sort_index(), y_train.sort_index()              # the training people in signup order (user_id rises with the signup time)
print([round(float(ys.iloc[test].mean()), 3) for _, test in KFold(5).split(Xs)])        # five blocks in time order, no shuffling
print([round(float(ys.iloc[test].mean()), 3) for _, test in folds.split(Xs, ys)])       # stratified folds
rows_ = []
for train_ix, test_ix in TimeSeriesSplit(n_splits=4).split(Xs):                          # each fold trains on the past and tests on what comes next
    m = logit().fit(Xs.iloc[train_ix], ys.iloc[train_ix])
    p = m.predict_proba(Xs.iloc[test_ix])[:, 1]
    rows_.append((len(train_ix), round(float(ys.iloc[test_ix].mean()), 3), round(float(p.mean()), 3), round(float(roc_auc_score(ys.iloc[test_ix], p)), 3)))
print(pd.DataFrame(rows_, columns=["trained on", "churned in test block", "mean chance", "AUC"]).to_string(index=False))
print(round(float(np.mean([r[3] for r in rows_])), 3))
gapped = []
for train_ix, test_ix in TimeSeriesSplit(n_splits=4, gap=1500).split(Xs):                # the same folds with 1,500 people left out between train and test
    gapped.append(round(float(roc_auc_score(ys.iloc[test_ix], logit().fit(Xs.iloc[train_ix], ys.iloc[train_ix]).predict_proba(Xs.iloc[test_ix])[:, 1])), 3))
print(gapped, round(float(np.mean(gapped)), 3))

## Unit 4: Learning inside each fold

**Predict first.** The 300 columns are random numbers: nothing in them predicts churn. The first route picks the 20 columns most related to churn from all 400 people and then cross-validates; the second puts the selection inside the folds. What AUC will each report: near 0.5, a little above, or well above?

In [ ]:
rng = np.random.default_rng(SEED)
pilot = X_train.sample(400, random_state=SEED)
y_pilot = y_train.loc[pilot.index].to_numpy()
noise = rng.normal(size=(400, 300))                   # 300 columns of random numbers: nothing in them predicts churn
five = StratifiedKFold(5, shuffle=True, random_state=SEED)
top20 = SelectKBest(f_classif, k=20).fit(noise, y_pilot)             # outside the folds: it looks at every label first
leaky = cross_val_score(LogisticRegression(max_iter=1000), top20.transform(noise), y_pilot, cv=five, scoring="roc_auc")
honest = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)), noise, y_pilot, cv=five, scoring="roc_auc")
print(round(float(leaky.mean()), 3), round(float(honest.mean()), 3))

## Finish it

### Exercise 1: Finish it

**Task:** **Do ten folds change the answer?** Score the day-7 model, `logit()`, on the ten stratified folds `ten` (given) of the training part with ROC AUC, and put the list [mean, lowest, highest] of the ten scores in `answer_1`, each rounded to 3 decimals.

**Expected:** `answer_1` is a list of three numbers.

In [ ]:
ten = StratifiedKFold(10, shuffle=True, random_state=SEED)       # given: ten folds
# score the day-7 model, logit(), on the ten folds of the training part with ROC AUC
answer_1 = None  # replace None with [mean, lowest, highest] of the ten scores, each rounded to 3 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", [0.692, 0.673, 0.73], traps=[([0.692, 0.683, 0.702], "That is the mean, the lowest and the highest of the five folds of unit 2. Use the ten folds.")], tol=5e-4)

<details>
<summary>Hint</summary>

`cross_val_score(logit(), X_train, y_train, cv=ten, scoring="roc_auc")` gives the ten scores; `mean`, `min` and `max` of the array give the three numbers.

</details>

<details>
<summary>Half-solution</summary>

```python
s10 = cross_val_score(..., X_train, y_train, cv=..., scoring="roc_auc")
answer_1 = [round(float(s10.mean()), 3), ..., ...]
```

</details>

<details>
<summary>Solution</summary>

```python
s10 = cross_val_score(logit(), X_train, y_train, cv=ten, scoring="roc_auc")
answer_1 = [round(float(s10.mean()), 3), round(float(s10.min()), 3), round(float(s10.max()), 3)]
```

Ten folds give a mean of 0.692, with scores from 0.673 to 0.730. The mean hardly moves from the five-fold mean of 0.692, and the range widens because each fold holds fewer people. More folds cost more fits and do not buy a better answer once the mean is stable.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Fewer, longer folds in time.** Before you start, say in one sentence what the result looks like: three AUC scores between 0.6 and 0.8. Use `Xs` and `ys`, the training people in signup order from unit 3. Score the day-7 model on the three folds of `TimeSeriesSplit(n_splits=3)`, each fold trained on the earlier people and tested on the next block, and put the list of the three AUC scores in `answer_2`, each rounded to 3 decimals.

**Expected:** `answer_2` is a list of three numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [0.646, 0.702, 0.726], traps=[([0.651, 0.667, 0.708], "That uses four splits, the folds of unit 3. Ask for three.")], tol=5e-4)

<details>
<summary>Hint</summary>

Loop over `TimeSeriesSplit(n_splits=3).split(Xs)`: fit `logit()` on the rows of `train_ix`, take the chances of the rows of `test_ix`, and score them with `roc_auc_score`.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"Three scores, each on a later and larger stretch, the first near 0.65."*

```python
for train_ix, test_ix in TimeSeriesSplit(n_splits=3).split(Xs):
    m = logit().fit(Xs.iloc[train_ix], ys.iloc[train_ix])
```

</details>

<details>
<summary>Solution</summary>

```python
ex2_scores = []
for train_ix, test_ix in TimeSeriesSplit(n_splits=3).split(Xs):
    m = logit().fit(Xs.iloc[train_ix], ys.iloc[train_ix])
    ex2_scores.append(round(float(roc_auc_score(ys.iloc[test_ix], m.predict_proba(Xs.iloc[test_ix])[:, 1])), 3))
answer_2 = ex2_scores
```

The three scores are 0.646, 0.702 and 0.726. With three splits each test block is longer (about 3,340 people) and the first model trains on 3,342 people, so the first score is the weakest. The mean, 0.691, says as before that the model holds up on later people.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague tunes a tree on a pilot of 1,000 people: 56 settings, each scored by cross-validation, and reports the best cross-validated score as the model's AUC. Change the last line so that the number is a score on people who played no part in the choice: fit the chosen model on the pilot and score it once on the test part. Put it in `answer_3`, rounded to 3 decimals.

**Expected:** `answer_3` is a number between 0.5 and 1.

In [ ]:
pilot3, y_pilot3 = X_train.iloc[:1000], y_train.iloc[:1000]            # given: a pilot of 1,000 people
best_score, best_model = -1.0, None
for depth in range(1, 15):
    for leaf in [1, 5, 20, 50]:                                       # given: 56 settings in all
        candidate = tree(depth, leaf)
        score = cross_val_score(candidate, pilot3, y_pilot3, cv=folds, scoring="roc_auc").mean()
        if score > best_score:
            best_score, best_model = score, candidate
answer_3 = round(float(best_score), 3)      # colleague: "the best of the 56 settings scores this, so that is the model's AUC"
# change the last line: fit the chosen model on the pilot and score it once on the test part

In [ ]:
_ = check("Exercise 3", "answer_3", 0.653, traps={0.668: "That is the best of the 56 cross-validated scores, the one that won the choice. Score the chosen model on the test part."}, tol=5e-4)

<details>
<summary>Hint</summary>

The cross-validated score of the winner is the best of 56 draws, so it is biased upward. Which rows have had no part in picking the winner?

</details>

<details>
<summary>Half-solution</summary>

```python
best_model.fit(pilot3, y_pilot3)
answer_3 = round(float(roc_auc_score(y_test, best_model.predict_proba(...)[:, 1])), 3)
```

</details>

<details>
<summary>Solution</summary>

```python
best_model.fit(pilot3, y_pilot3)
answer_3 = round(float(roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])), 3)
```

The winner of the 56 scores 0.668 in cross-validation (depth 4, at least 20 people in each leaf) and 0.653 on the test part. The gap is the price of choosing the maximum of 56 noisy scores: the best score was picked partly because it was lucky. The test part gives the honest number, and it can be used once.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")